# 10A · The Simulation Engine — Ten Thousand Futures From First Principles
### Financial Analytics — Module 10 · Lab 2

**Monte Carlo in one sentence:** when the mathematics of uncertainty gets hard, *play the game many times and count what happens.*

Named for the casino, used everywhere hard probability lives — physics, epidemiology, and finance's biggest questions: will my money last? what could this portfolio lose? is this project worth it? Today you build the engine; 10B and 10C point it at those questions.

> 🛡️ **Bias check:** simulations inherit their inputs' biases. Drift and volatility below are estimated from NIFTY history — a window containing the registry's regime shift (noted: our vol estimate is a blend of two regimes). No survivorship (index data), no restatements. Look-ahead: none — we simulate forward only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)     # THE SEED. Same seed = same futures = reproducible analysis.
                                     # Change it and every number below changes slightly - that's not error, that's the method.

---
## 1. A random walk from a coin flip

Start stupid-simple: each day, a price moves +1 or −1 by coin flip. That's it — and watch what emerges:

In [ ]:
flips = rng.choice([-1, 1], size=(300, 250))     # 300 walks, 250 "days" each
walks = flips.cumsum(axis=1)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(walks[:60].T, lw=0.5, alpha=0.4)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("300 coin-flip walks (60 shown): pure 50/50 chance - yet look at the SPREAD grow",
             loc="left", fontweight="bold")
ax.set_xlabel("day"); plt.tight_layout(); plt.show()

spread = walks.std(axis=0)
print(f"Spread after  25 days: {spread[24]:.1f}")
print(f"Spread after 100 days: {spread[99]:.1f}   (4x the time...)")
print(f"Spread after 225 days: {spread[224]:.1f}   (9x the time...)")
print(f"\nRatios: {spread[99]/spread[24]:.2f}x and {spread[224]/spread[24]:.2f}x -> spread grows with SQUARE ROOT of time.")

**There it is — the promise from Module 3, kept.** Uncertainty grows with **√time**: 4× the horizon, only 2× the spread; 9× the horizon, 3× the spread. That's why annualising daily volatility multiplies by √252 — you just *derived* it by counting, no algebra required. This is Monte Carlo's superpower: results you'd otherwise have to prove, you can simply *observe*.

---
## 2. From coin flips to a market: drift + volatility

Real returns aren't ±1. They're draws with a **drift** (the average daily pull) and a **volatility** (the daily wobble) — both estimable from history:

In [ ]:
BASE = "data/"
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
rets = px["close"].pct_change().dropna()

mu, sigma = rets.mean(), rets.std()
print(f"Estimated daily drift mu    = {mu:.5f}  (~{mu*252:.1%} per year)")
print(f"Estimated daily vol   sigma = {sigma:.5f}  (~{sigma*np.sqrt(252):.1%} per year)")
print(f"Starting price: {px['close'].iloc[-1]:,.0f}")

In [ ]:
# Simulate: each future day's return = mu + sigma x (random normal draw)
# Prices COMPOUND (multiplicative), so we apply returns with cumprod - a price can fall 50% then rise 50%
# and NOT be back where it started. Multiplicative worlds also can't go below zero. (This structure -
# normal returns compounding multiplicatively - is the intuition behind 'geometric Brownian motion',
# the standard textbook model. You've now built it, which beats naming it.)
N_PATHS, N_DAYS = 10_000, 252
S0 = px["close"].iloc[-1]

sim_rets  = mu + sigma * rng.standard_normal((N_PATHS, N_DAYS))
sim_paths = S0 * np.cumprod(1 + sim_rets, axis=1)

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.plot(sim_paths[:150].T, lw=0.4, alpha=0.3, color="#2563EB")
for q, clr, lab in [(0.05, "#DC2626", "5th pct"), (0.50, "black", "median"), (0.95, "#16A34A", "95th pct")]:
    ax.plot(np.quantile(sim_paths, q, axis=0), color=clr, lw=2, label=lab)
ax.set_title(f"10,000 simulated NIFTY years (150 shown) - Module 9's cone, now built from scratch",
             loc="left", fontweight="bold")
ax.set_xlabel("trading day"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

final = sim_paths[:, -1]
print(f"One year out: median {np.median(final):,.0f} | 5th pct {np.quantile(final, .05):,.0f} | 95th pct {np.quantile(final, .95):,.0f}")
print(f"P(ending below today) = {(final < S0).mean():.1%}  <- positive drift, yet down-years remain common")

**Read the cone you just built.** The median drifts up (the mu at work); the 5th–95th band widens with √t (the sigma at work); and even with positive drift, a meaningful share of paths end *down* — a truth about equity investing that no average return communicates, and exactly what Monte Carlo exists to show.

---
## 3. How many simulations is enough? Monte Carlo error

In [ ]:
# Any Monte Carlo answer is an ESTIMATE with its own noise. Watch the estimate settle as N grows:
target = (final < S0).mean()      # our "true" (N=10,000) estimate of P(down year)
for n in [100, 1_000, 10_000]:
    estimates = [(sim_paths[rng.choice(N_PATHS, n), -1] < S0).mean() for _ in range(200)]
    print(f"N={n:>6,}: P(down) estimates range {np.min(estimates):.3f} - {np.max(estimates):.3f}  (spread {np.std(estimates):.4f})")
print("\nEach 10x in N shrinks the noise ~3.2x (that's 1/sqrt(N) - the square root again!).")
print("Practical rule: 10,000 paths for percentiles, more if you care about the extreme 1% tail.")

### ✏️ Exercises
1. **The seed experiment:** re-run the whole simulation with seed 7. Which numbers change, and by how much? Which *conclusions* change? Write the one-sentence policy this suggests for reporting Monte Carlo results.
2. **Two regimes, two cones:** estimate mu and sigma separately from the calm early period (2021–22) and the choppier rest. Simulate a year with each. How different are the 5th percentiles — and which cone would you show a risk committee? (There's no clean answer; there IS a required sentence about regime assumptions.)
3. **Break the multiplicative rule:** simulate with ADDITIVE moves (`S0 + sim_rets.cumsum(axis=1)*S0`). Run 10 years instead of 1. What impossible thing starts happening to some paths, and why does compounding prevent it?

---
*AI disclosure: ______*

In [ ]:
# workspace
